# USD Export with `bpy_lattice`

This notebook demonstrates how to export a particle-accelerator lattice to
[Universal Scene Description (USD)](https://openusd.org/) using `bpy_lattice`.

The resulting `.usda` / `.usd` files can be imported into
**Blender**, **NVIDIA Omniverse**, or any other USD-compatible application.

**Key features:**

- Prototype instancing — geometry is defined once and shared across elements
- `UsdPreviewSurface` materials for correct rendering in Omniverse and Blender
- Track export as `BasisCurves`
- Optional CAD-model integration from a Blender Catalogue

## 1 — Imports

In [ ]:
from pytao import Tao

from bpy_lattice import Lattice

## 2 — Define a lattice in Bmad

We define a compact FODO storage ring inline so the notebook is entirely
self-contained.  Eight identical cells, each with focusing / defocusing
quadrupoles, sextupoles, drifts, and dipole bends that close the ring
in 360°.

In [ ]:
LATTICE_STR = """
no_digested

parameter[geometry] = closed
parameter[particle] = electron
beginning[e_tot] = 3e9

! ---- Element definitions ----
qf: quadrupole, l = 0.3, k1 = 1.2
qd: quadrupole, l = 0.3, k1 = -1.2
sf: sextupole,  l = 0.2, k2 = 50
sd: sextupole,  l = 0.2, k2 = -50
b:  sbend,      l = 1.0, angle = pi/8
d:  drift,      l = 0.5

! ---- Apertures ----
qf[x_limit] = 0.04
qf[y_limit] = 0.02
qd[x_limit] = 0.04
qd[y_limit] = 0.02
b[x_limit]  = 0.05
b[y_limit]  = 0.025

! ---- Lattice layout ----
fodo: line = (qf, d, b, d, sf, qd, d, b, d, sd)
ring: line = (8*fodo)
use, ring
"""

## 3 — Create a Tao instance and build the Lattice

In [ ]:
tao = Tao.from_lattice_contents(LATTICE_STR, plot="mpl")
tao.plot()

In [ ]:
lattice = Lattice.from_tao(tao)

print(f"Elements: {len(lattice.elements)}")
print(f"Tracks  : {len(lattice.tracks)}")

## 4 — Export to USD

A single call produces the `.usda` file.  The returned `Usd.Stage` can
be used for programmatic inspection.

In [ ]:
stage = lattice.to_usd("fodo_ring.usda")

The file `fodo_ring.usda` is now on disk.  You can open it directly in
**Blender** (*File → Import → USD*) or drag it into **NVIDIA Omniverse**.

## 5 — Inspect the stage

In [ ]:
from pxr import UsdGeom

all_prims = list(stage.Traverse())
meshes = [p for p in all_prims if p.IsA(UsdGeom.Mesh)]
xforms = [p for p in all_prims if p.IsA(UsdGeom.Xform)]
curves = [p for p in all_prims if p.IsA(UsdGeom.BasisCurves)]

print(f"Total prims : {len(all_prims)}")
print(f"Xform prims : {len(xforms)}")
print(f"Mesh prims  : {len(meshes)}")
print(f"Curve prims : {len(curves)}")
print(f"Up axis     : {UsdGeom.GetStageUpAxis(stage)}")
print(f"Meters/unit : {UsdGeom.GetStageMetersPerUnit(stage)}")

### Scene hierarchy

In [ ]:
def print_tree(prim, depth=0, max_depth=3):
    if depth > max_depth:
        return
    indent = "  " * depth
    typ = prim.GetTypeName() or "(no type)"
    print(f"{indent}{prim.GetName()}  [{typ}]")
    for child in prim.GetChildren():
        print_tree(child, depth + 1, max_depth)


print_tree(stage.GetDefaultPrim())

### Prototype sharing

Identical geometry is defined once under `/Root/Prototypes` and referenced
by every element of the same type / shape — keeping the file small and
rendering fast.

In [ ]:
proto_scope = stage.GetPrimAtPath("/Root/Prototypes")
print("Prototypes:")
for child in proto_scope.GetChildren():
    print(f"  {child.GetName():45s}  [{child.GetTypeName()}]")

### Element metadata

Each element prim stores the original name, class, and description as
`customData`.

In [ ]:
# Pick the first Bend
bends_scope = stage.GetPrimAtPath("/Root/Elements/Bends")
first_bend = next(iter(bends_scope.GetChildren()))
print(f"Prim path  : {first_bend.GetPath()}")
print(f"Custom data: {dict(first_bend.GetCustomData())}")

## 6 — Export options

| Parameter | Default | Description |
|---|---|---|
| `filepath` | `"lattice.usda"` | Output path (`.usda` = ASCII, `.usd`/`.usdc` = binary) |
| `up_axis` | `"Y"` | Stage up-axis (`"Y"` for Omniverse, `"Z"` for Blender) |
| `meters_per_unit` | `1.0` | Scale factor (1.0 = positions in metres) |
| `catalogue` | `None` | Path to a CAD-model catalogue directory |
| `blender_cmd` | `None` | Blender executable for `.blend` → `.usd` conversion (auto-detected) |
| `copy_models` | `True` | Copy catalogue USD models locally (`False` to reference in place) |

### Binary (crate) format

Use `.usd` or `.usdc` for a compact binary file:

In [ ]:
import os

lattice.to_usd("fodo_ring.usd")

ascii_kb = os.path.getsize("fodo_ring.usda") / 1024
binary_kb = os.path.getsize("fodo_ring.usd") / 1024
print(f"ASCII  : {ascii_kb:.1f} KB")
print(f"Binary : {binary_kb:.1f} KB")
print(f"Ratio  : {binary_kb / ascii_kb:.1%}")

### Z-up axis (Blender convention)

In [ ]:
stage_z = lattice.to_usd("fodo_ring_z_up.usda", up_axis="Z")
print(f"Up axis: {UsdGeom.GetStageUpAxis(stage_z)}")

### Functional API

`lattice_to_usd` works directly with element / track lists:

In [ ]:
from bpy_lattice import lattice_to_usd

stage_fn = lattice_to_usd(
    elements=lattice.elements,
    tracks=lattice.tracks,
    filepath="fodo_ring_functional.usda",
)
print(f"Prims: {len(list(stage_fn.Traverse()))}")

### JSON round-trip

Serialize to JSON and back, then export to USD:

In [ ]:
lattice.to_json("fodo_ring.json")

lattice2 = Lattice.from_json("fodo_ring.json")
lattice2.to_usd("fodo_ring_from_json.usda")

print(f"Elements: {len(lattice2.elements)}")
print(f"Tracks  : {len(lattice2.tracks)}")

## 7 — Catalogue integration

When elements have a `cad_model` field pointing to a `.blend` or `.usd`
file, pass a `catalogue` directory so those models are resolved and
referenced automatically:

```python
lattice.to_usd(
    "lattice_with_cad.usda",
    catalogue="/path/to/Catalogue",
)
```

`.blend` files are converted to `.usd` via Blender in headless mode
(auto-detected).  Converted models are cached in a `models/` subdirectory
next to the output file — subsequent exports reuse them instantly.

Set `copy_models=False` to reference USD catalogue files in place
rather than copying them:

```python
lattice.to_usd(
    "lattice_with_cad.usda",
    catalogue="/path/to/Catalogue",
    copy_models=False,
)
```

## Cleanup

In [ ]:
from pathlib import Path

for f in Path(".").glob("fodo_ring*"):
    f.unlink()
    print(f"Removed {f}")